# Named-entity recognition: discriminative and generative formulations

The same 64 WikiANN validation examples are evaluated with a pretrained BERT token classifier and an instruction-tuned causal LM. Both outputs are mapped to word-level BIO tags and scored with exact entity-level precision, recall and F1.

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from seqeval.metrics import classification_report, f1_score

from nlp_portfolio.ner import (pipeline_entities_to_dict, text_entities_to_bio,
                               wikiann_gold_tags, align_word_labels)

MODEL_ID = "dslim/bert-base-NER"
dataset = load_dataset("wikiann", "en", split="validation[:64]")
label_names = dataset.features["ner_tags"].feature.names
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(MODEL_ID)
recognizer = pipeline("token-classification", model=model, tokenizer=tokenizer,
                      aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)
print("examples:", len(dataset), "labels:", label_names)

## Subword-label alignment

Special tokens receive `-100` so cross-entropy ignores them. When a word with `B-X` is split, subsequent pieces become `I-X`; otherwise a single entity would be counted as several beginnings.

In [ ]:
row = dataset[0]
encoded = tokenizer(row["tokens"], is_split_into_words=True)
label_to_id = {name: index for index, name in enumerate(label_names)}
aligned = align_word_labels(encoded.word_ids(), row["ner_tags"], label_names, label_to_id)
pd.DataFrame({
    "token": tokenizer.convert_ids_to_tokens(encoded["input_ids"]),
    "word_id": encoded.word_ids(),
    "training_label": aligned,
}).head(20)

In [ ]:
gold, bert_predictions = [], []
for item in dataset:
    text = " ".join(item["tokens"])
    entities = pipeline_entities_to_dict(recognizer(text))
    gold.append(wikiann_gold_tags(item["ner_tags"], label_names))
    bert_predictions.append(text_entities_to_bio(item["tokens"], entities))

print("BERT entity-level F1:", round(f1_score(gold, bert_predictions), 4))
print(classification_report(gold, bert_predictions, digits=3))

## Prompted extraction with an instruction model

The generator returns JSON arrays of exact substrings. Invalid JSON becomes an empty prediction, so formatting failures reduce recall rather than being silently repaired with gold information.

In [ ]:
RUN_GENERATIVE_NER = False
if RUN_GENERATIVE_NER:
    from transformers import AutoModelForCausalLM
    from nlp_portfolio.ner import predict_chat_entities

    GEMMA_ID = "unsloth/gemma-3-1B-it"
    chat_tokenizer = AutoTokenizer.from_pretrained(GEMMA_ID)
    chat_model = AutoModelForCausalLM.from_pretrained(
        GEMMA_ID, device_map="auto",
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    chat_predictions, raw_answers = [], []
    for item in dataset:
        entities, raw = predict_chat_entities(" ".join(item["tokens"]), chat_tokenizer, chat_model)
        chat_predictions.append(text_entities_to_bio(item["tokens"], entities))
        raw_answers.append(raw)
    print("Gemma entity-level F1:", round(f1_score(gold, chat_predictions), 4))

## Boundary-level error audit

Exact entity scoring penalizes type errors and boundary errors. The table below records the first disagreement in each sentence, which helps distinguish tokenization problems from missing entities.

In [ ]:
errors = []
for item, expected, predicted in zip(dataset, gold, bert_predictions):
    mismatch = [i for i, (a, b) in enumerate(zip(expected, predicted)) if a != b]
    if mismatch:
        i = mismatch[0]
        errors.append({"text": " ".join(item["tokens"]), "token": item["tokens"][i],
                       "gold": expected[i], "prediction": predicted[i]})
pd.DataFrame(errors).head(12)

The pretrained checkpoint and WikiANN do not share exactly the same annotation policy. The comparison therefore evaluates transfer under label mapping, not supervised fine-tuning. A full fine-tuning experiment should align labels before training and reserve a separate test split for the final score.